## AgentCore Runtime의 Progress Notifications

이 튜토리얼에서는 MCP specification에 정의된 MCP utility인 **Progress Notifications**를 보여줍니다. progress notification을 사용하면 장시간 tool 실행 중 **server**가 실시간 update를 client에 streaming할 수 있어 사용자가 기다리기만 하지 않고 진행 상황을 확인할 수 있습니다.

### Progress Notifications란?

표준 MCP에서는 client가 tool을 호출하고 결과를 기다리며, server가 응답할 때까지 내부 동작을 알 수 없습니다. progress notification을 사용하면 server가 **실행 도중 update를 전송**하여 client가 각 작업 단계를 확인할 수 있습니다.

이는 다른 stateful MCP primitive와 다음과 같은 차이가 있습니다.
- **Elicitation**: server가 client에 **user input** 요청(block하고 response 대기)
- **Sampling**: server가 client에 **LLM inference** 요청(block하고 response 대기)
- **Progress**: server가 client에 **execution update** 전송 - fire-and-forget 방식으로 response 불필요

### Fire-and-Forget과 Request/Response 비교

```
Elicitation / Sampling:
  Server ──request──▶ Client
  Server ◀──response── Client   ← server가 여기서 대기
  Server 계속 실행...

Progress:
  Server ──notification──▶ Client   ← client가 update rendering
  Server 즉시 계속 실행...           ← 대기 없음
  Server ──notification──▶ Client
  Server continues immediately...
  ...
```

### Progress Notifications 사용 시점

- 완료하는 데 1~2초 이상 걸리는 모든 tool
- 각 단계가 사용자에게 의미 있는 multi-step workflow
- batch 작업(데이터 가져오기, record 처리)
- client가 progress bar 또는 상태를 rendering하도록 하려는 경우

### 튜토리얼 세부 정보

| 항목 | 세부 정보 |
|:------------|:--------|
| 튜토리얼 유형 | Stateful MCP — Progress Notifications |
| 기능 | `ctx.report_progress()` — server-to-client execution update |
| Session mode | Stateful(`stateless_http=False`) |
| Client | `progress_handler`가 포함된 `fastmcp.Client` |
| 사용 사례 | 5단계 월간 재무 보고서(finance tracker) |

In [ ]:
!pip install -qU -r requirements.txt

script가 helpers 폴더에 액세스할 수 있도록 다음 path를 추가합니다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

### Progress MCP Server 작성

server는 **`generate_report`**라는 하나의 tool을 노출합니다. 이 tool은 서로 다른 5단계로 월간 재무 보고서를 작성하며 각 단계 시작 시 `ctx.report_progress()`를 호출하여 client가 실행을 실시간으로 추적할 수 있도록 합니다.

| 단계 | progress/total | 수행 작업 |
|:-----|:--------------|:----------|
| 1 | 1/5 | DynamoDB에서 모든 transaction 가져오기 |
| 2 | 2/5 | category별 지출 grouping 및 합계 계산 |
| 3 | 3/5 | DynamoDB에서 budget limit 가져오기 |
| 4 | 4/5 | 지출과 budget 비교 |
| 5 | 5/5 | 최종 보고서 형식 지정 및 반환 |

stateless server와의 주요 차이점은 다음과 같습니다.
- `stateless_http`를 **설정하지 않음**(기본값 `False`) - progress notification에 필요
- `generate_report`는 `async def`이며 `ctx: Context`를 받음
- `await ctx.report_progress(progress=N, total=5)` 사용 - fire-and-forget 방식으로 실행이 즉시 계속됨
- 단계 사이의 `asyncio.sleep()`으로 Notebook에서 진행 상황 확인 가능

In [ ]:
%%writefile agents/mcp_progress_server.py
import os
import asyncio
from fastmcp import FastMCP, Context
from dynamo_utils import FinanceDB

mcp = FastMCP(name='Progress-MCP-Server')

_region = os.environ.get('AWS_REGION') or os.environ.get('AWS_DEFAULT_REGION') or 'us-east-1'
db = FinanceDB(region_name=_region)

@mcp.tool()
async def generate_report(user_alias: str, ctx: Context) -> str:
    """Generate a monthly financial report in 5 steps, streaming a progress
    notification to the client at the start of each stage.

    Args:
        user_alias: User identifier
    """
    total = 5

    # 1단계: transaction 가져오기
    await ctx.report_progress(progress=1, total=total)
    await asyncio.sleep(0.5)
    transactions = db.get_transactions(user_alias)
    if not transactions:
        return f'No transactions found for {user_alias}.'

    # 2단계: category별 grouping
    await ctx.report_progress(progress=2, total=total)
    await asyncio.sleep(0.5)
    by_category = {}
    for t in transactions:
        cat = t['category']
        by_category[cat] = by_category.get(cat, 0) + abs(float(t['amount']))

    # 3단계: budget 가져오기
    await ctx.report_progress(progress=3, total=total)
    await asyncio.sleep(0.5)
    budgets = {b['category']: float(b['monthly_limit']) for b in db.get_budgets(user_alias)}

    # 4단계: 지출과 budget 비교
    await ctx.report_progress(progress=4, total=total)
    await asyncio.sleep(0.5)
    lines = []
    for cat, spent in sorted(by_category.items(), key=lambda x: -x[1]):
        limit = budgets.get(cat)
        if limit:
            pct = (spent / limit) * 100
            status = 'OVER' if spent > limit else 'OK'
            lines.append(f'  {cat:<15} ${spent:>8.2f} / ${limit:.2f}  [{pct:.0f}%] {status}')
        else:
            lines.append(f'  {cat:<15} ${spent:>8.2f}  (no budget set)')

    # 5단계: 보고서 형식 지정
    await ctx.report_progress(progress=5, total=total)
    await asyncio.sleep(0.2)
    total_spent = sum(by_category.values())
    report = (
        f'Monthly Report for {user_alias}\n'
        f'{"=" * 50}\n'
        f'  {"Category":<15} {"Spent":>10}   {"Budget":>8}  Status\n'
        f'{"-" * 50}\n'
        + '\n'.join(lines) +
        f'\n{"-" * 50}\n'
        f'  {"TOTAL":<15} ${total_spent:>8.2f}\n'
    )
    return report


if __name__ == '__main__':
    mcp.run(
        transport="streamable-http",
        host="0.0.0.0",
        port=8000,
        stateless_http=False
    )

In [ ]:
%%writefile agents/requirements.txt
fastmcp>=2.10.0
mcp
bedrock-agentcore

`dynamo_utils` 파일을 agents 폴더로 복사합니다.

In [ ]:
!cp ../helpers/dynamo_utils.py agents/dynamo_utils.py

### DynamoDB Table 및 Seed Data 생성

보고서에서 의미 있는 데이터를 사용할 수 있도록 `finance_tracker` table을 생성하고 sample expense와 budget을 seed합니다.

In [ ]:
import boto3

from helpers.dynamo_utils import FinanceDB

region = boto3.session.Session().region_name
db = FinanceDB(region_name=region)
result = db.create_table()
print(f"Region: {region}")
print(result)

In [ ]:
# expense seed 데이터
print("Seeding expenses...")
for user, amount, desc, cat in [
    ("me", 45.50, "Dinner", "food"),
    ("me", 85.30, "Groceries", "food"),
    ("me", 120.00, "Electricity", "bills"),
    ("me", 55.00, "Gas", "transport"),
    ("me", 15.99, "Netflix", "entertainment"),
    ("me", 30.00, "Spotify + apps", "entertainment"),
]:
    print(f"  {db.add_transaction(user, 'expense', -abs(amount), desc, cat)}")

# budget seed 데이터
print("\nSeeding budgets...")
for user, cat, limit in [
    ("me", "food", 100.00),
    ("me", "bills", 100.00),
    ("me", "transport", 60.00),
    ("me", "entertainment", 40.00),
]:
    print(f"  {db.set_budget(user, cat, limit)}")

### Cognito Authentication 설정

In [ ]:
from helpers.utils import get_or_create_cognito_pool, reauthenticate_user

print("Setting up Amazon Cognito user pool...")
cognito_config = get_or_create_cognito_pool()
print("Cognito setup completed ✓")

### AgentCore Execution Role 생성

In [ ]:
from helpers.utils import create_agentcore_runtime_execution_role, SAMPLE_ROLE_NAME

execution_role_arn = create_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)

### AgentCore Runtime에 배포

In [ ]:
import boto3
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = Session()
sts = boto3.client("sts")
account_id = sts.get_caller_identity()["Account"]

aws_agent_name = "mcp_progress_server"
runtime = Runtime()

runtime.configure(
    entrypoint="agents/mcp_progress_server.py",
    execution_role=execution_role_arn,
    auto_create_ecr=True,
    requirements_file="agents/requirements.txt",
    region=region,
    agent_name=aws_agent_name,
    authorizer_configuration={
        "customJWTAuthorizer": {
            "allowedClients": [cognito_config.get("client_id")],
            "discoveryUrl": cognito_config.get("discovery_url"),
        }
    },
    protocol="MCP",
    deployment_type="direct_code_deploy",
    runtime_type="PYTHON_3_13",
)

launch_result = runtime.launch()
print("Launch completed:", launch_result.agent_arn)

In [ ]:
status_response = runtime.status()
print(f"Final status: {status_response.endpoint['status']}")

### Progress Notifications 테스트

배포된 server를 호출하고 progress bar가 5단계에 걸쳐 움직이는 것을 확인합니다.

#### progress_handler 작동 방식

server가 `ctx.report_progress()`를 호출할 때마다 fastmcp.Client는 `notifications/progress` message를 받고 **server를 block하지 않은 채** 즉시 `progress_handler`를 호출합니다. handler의 rendering이 끝날 때는 server가 이미 다음 단계를 실행 중입니다.

| 단계 | Server 전송 | Handler rendering |
|:-----|:------------|:----------------|
| 1 | `progress=1, total=5` | `[####----------------] 20%` |
| 2 | `progress=2, total=5` | `[########------------] 40%` |
| 3 | `progress=3, total=5` | `[############--------] 60%` |
| 4 | `progress=4, total=5` | `[################----] 80%` |
| 5 | `progress=5, total=5` | `[####################] 100%  Done!` |

In [ ]:
ac_runtime_name = launch_result.agent_id

mcp_url = (
    f"https://bedrock-agentcore.{region}.amazonaws.com"
    f"/runtimes/{ac_runtime_name}/invocations"
    f"?qualifier=DEFAULT&accountId={account_id}"
)

bearer_token = reauthenticate_user(cognito_config.get("client_id"), cognito_config.get("client_secret"))

headers = {
    "authorization": f"Bearer {bearer_token}",
    "Content-Type": "application/json",
}

print(f"MCP URL: {mcp_url}")

In [ ]:
async def progress_handler(progress: float, total: float | None, message: str | None):
    """각 진행 알림을 실시간 ASCII 진행률 표시줄로 렌더링합니다."""
    pct = int((progress / total) * 100) if total else 0
    filled = pct // 5
    bar = "#" * filled + "-" * (20 - filled)
    print(
        f"\r  Progress: [{bar}] {pct}% ({int(progress)}/{int(total or 0)})",
        end="",
        flush=True,
    )
    if total and progress >= total:
        print("  Done!")

In [ ]:
import logging

logging.getLogger("mcp.client.streamable_http").setLevel(logging.ERROR)

from fastmcp import Client
from fastmcp.client.transports import StreamableHttpTransport

transport = StreamableHttpTransport(url=mcp_url, headers=headers)

print("Testing Progress Notifications...\n" + "=" * 50)
async with Client(transport, progress_handler=progress_handler) as client:
    result = await client.call_tool("generate_report", {"user_alias": "me"})

print("=" * 50)
print(f"\n{result.content[0].text}")

---

### 리소스 정리(선택 사항)

아래 셀을 실행하여 이 튜토리얼에서 생성한 모든 AWS 리소스를 삭제합니다.

In [ ]:
from pathlib import Path
from bedrock_agentcore_starter_toolkit.operations.runtime.destroy import (
    destroy_bedrock_agentcore,
)

print("Destroying AgentCore runtime...")
destroy_bedrock_agentcore(config_path=Path(".bedrock_agentcore.yaml"), agent_name=aws_agent_name)

In [ ]:
from helpers.utils import delete_agentcore_runtime_execution_role

print("Deleting IAM execution role...")
delete_agentcore_runtime_execution_role(SAMPLE_ROLE_NAME)
print("Execution role deleted ✓")

In [ ]:
from helpers.utils import cleanup_cognito_resources, delete_cognito_secret

print("Cleaning up Cognito resources...")
cleanup_cognito_resources(cognito_config.get("pool_id"))
print("Cognito resources cleaned up ✓")

print("Deleting Cognito secret...")
delete_cognito_secret()
print("Secret deleted ✓")

In [ ]:
print("Deleting DynamoDB table...")
result = db.delete_table()
print(result)

In [ ]:
from helpers.utils import local_file_cleanup

print("📁 Starting Local Files cleanup...")
local_file_cleanup()